# Amazon Reviews'23 Video Games - RecSys Experiment (Colab)

This notebook: (1) bootstraps Colab (Drive, work dir, clone repo), (2) downloads the Video_Games dataset, (3) preprocesses, trains an MLP, and reports metrics.

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Setup working directory in Google Drive
import os
assert os.path.exists('/content/drive')
WORK_DIR = '/content/drive/MyDrive/colab/amazon_review_game'
%mkdir -p $WORK_DIR
%cd $WORK_DIR

In [ ]:
# Clone repo, install dependencies, and make src importable (Colab-friendly)
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

repo_url = 'https://github.com/allyoushawn/recsys_playground.git'
repo_dir = 'recsys_playground'
branch_name = 'main'

import os
if IN_COLAB:
    if os.path.exists(repo_dir):
      !rm -rf {repo_dir}
    !git clone $repo_url
    %cd $repo_dir
    !git fetch --all
    !git checkout $branch_name || echo 'Branch not found; staying on default.'

In [ ]:
# Install dependencies (omit jupyter - already in Colab)
!pip install -q torch pandas numpy scikit-learn matplotlib seaborn requests

In [ ]:
# Config
DATASET_URL = 'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Video_Games.jsonl.gz'
PROJECT_NAME = 'amazon_review_game'
force_rewrite = False
TASK_TYPE = 'ranking'  # 'ranking' or 'regression'
SAMPLE_SIZE = 200_000  # Use 200k rows for Colab; set None for full


In [ ]:
# Download dataset
import os
import urllib.request

DATA_DIR = f'/content/drive/MyDrive/colab/data/{PROJECT_NAME}'
os.makedirs(DATA_DIR, exist_ok=True)

reviews_file = os.path.join(DATA_DIR, 'Video_Games.jsonl.gz')
should_download = force_rewrite or not os.path.exists(reviews_file) or os.path.getsize(reviews_file) == 0

if should_download:
    print(f'Downloading to {reviews_file}...')
    urllib.request.urlretrieve(DATASET_URL, reviews_file)
    print('Done.')
else:
    print(f'Using existing data at {reviews_file}')

In [ ]:
# Inspect
import gzip
import json
import pandas as pd

def load_jsonl_gz(path, max_rows=None):
    rows = []
    with gzip.open(path, 'rt', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if max_rows and i >= max_rows:
                break
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return pd.DataFrame(rows)

df = load_jsonl_gz(reviews_file, max_rows=SAMPLE_SIZE)
print('Shape:', df.shape)
print('\nDtypes:')
print(df.dtypes)
print('\nHead:')
df.head()

In [ ]:
# Task detection
# user_id, parent_asin, rating -> ranking (or regression)
task = TASK_TYPE
print(f'Task: {task}')
print('Columns: user_id, item_id (parent_asin), rating, timestamp')

In [ ]:
# Preprocess
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Prepare interaction table
inter_df = df[['user_id', 'parent_asin', 'rating']].copy()
inter_df = inter_df.rename(columns={'parent_asin': 'item_id'})
inter_df = inter_df.dropna()
inter_df['rating'] = pd.to_numeric(inter_df['rating'], errors='coerce')
inter_df = inter_df.dropna()

# Encode user and item IDs
user_enc = LabelEncoder()
item_enc = LabelEncoder()
inter_df['user_idx'] = user_enc.fit_transform(inter_df['user_id'].astype(str))
inter_df['item_idx'] = item_enc.fit_transform(inter_df['item_id'].astype(str))

n_users = inter_df['user_idx'].nunique()
n_items = inter_df['item_idx'].nunique()
print(f'Users: {n_users}, Items: {n_items}, Interactions: {len(inter_df)}')

# Train/test split (80/20)
train_df, test_df = train_test_split(inter_df, test_size=0.2, random_state=42)
print(f'Train: {len(train_df)}, Test: {len(test_df)}')

In [ ]:
# Model + train
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

class RecMLP(nn.Module):
    def __init__(self, n_users, n_items, embed_dim=32, hidden_dim=64):
        super().__init__()
        self.user_emb = nn.Embedding(n_users + 1, embed_dim, padding_idx=0)
        self.item_emb = nn.Embedding(n_items + 1, embed_dim, padding_idx=0)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, user_idx, item_idx):
        u = self.user_emb(user_idx + 1)  # +1 for padding_idx
        i = self.item_emb(item_idx + 1)
        x = torch.cat([u, i], dim=1)
        return self.mlp(x).squeeze(-1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RecMLP(n_users, n_items).to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

# Data
X_u = torch.LongTensor(train_df['user_idx'].values)
X_i = torch.LongTensor(train_df['item_idx'].values)
y = torch.FloatTensor(train_df['rating'].values)
loader = DataLoader(TensorDataset(X_u, X_i, y), batch_size=1024, shuffle=True)

epochs = 10
losses = []
for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    for u, i, r in loader:
        u, i, r = u.to(device), i.to(device), r.to(device)
        opt.zero_grad()
        pred = model(u, i)
        loss = criterion(pred, r)
        loss.backward()
        opt.step()
        epoch_loss += loss.item()
    avg = epoch_loss / len(loader)
    losses.append(avg)
    print(f'Epoch {epoch+1}/{epochs} loss={avg:.4f}')

print('Training done.')

In [ ]:
# Report
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Training curve
plt.figure(figsize=(8, 4))
plt.plot(losses, marker='o', markersize=4)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training Curve')
plt.grid(True)
plt.tight_layout()
plt.show()

# Test metrics
model.eval()
with torch.no_grad():
    u_test = torch.LongTensor(test_df['user_idx'].values).to(device)
    i_test = torch.LongTensor(test_df['item_idx'].values).to(device)
    pred = model(u_test, i_test).cpu().numpy()

y_test = test_df['rating'].values
mse = mean_squared_error(y_test, pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)

print('Test Metrics:')
print(f'  MSE: {mse:.4f}')
print(f'  RMSE: {rmse:.4f}')
print(f'  MAE: {mae:.4f}')
print(f'  R²: {r2:.4f}')

# Simple ranking (NDCG, MRR, Hit Rate) - sample per user, batched scoring
def eval_ranking_at_k(model, test_df, k=10, n_users=500, batch_size=2000):
    model.eval()
    users = test_df['user_idx'].unique()[:n_users]
    ndcg_sum, mrr_sum, hit_sum = 0.0, 0.0, 0.0
    with torch.no_grad():
        for u in users:
            mask = test_df['user_idx'] == u
            items = test_df.loc[mask, 'item_idx'].values
            if len(items) == 0:
                continue
            gt = items[0]
            scores_list = []
            for start in range(0, n_items, batch_size):
                end = min(start + batch_size, n_items)
                u_t = torch.LongTensor([u] * (end - start)).to(device)
                all_items = torch.LongTensor(range(start, end)).to(device)
                s = model(u_t, all_items).cpu().numpy()
                scores_list.append(s)
            scores = np.concatenate(scores_list)
            top = np.argsort(-scores)[:k]
            if gt in top:
                hit_sum += 1
                rank = np.where(top == gt)[0][0] + 1
                mrr_sum += 1.0 / rank
                ndcg_sum += 1.0 / np.log2(rank + 1)
    n = len(users)
    return hit_sum / n if n else 0, mrr_sum / n if n else 0, ndcg_sum / n if n else 0

hit, mrr, ndcg = eval_ranking_at_k(model, test_df, k=10, n_users=min(500, n_users))
print(f'\nRanking (sample of users):')
print(f'  Hit@10: {hit:.4f}')
print(f'  MRR@10: {mrr:.4f}')
print(f'  NDCG@10: {ndcg:.4f}')